<a href="https://colab.research.google.com/github/theharshk/Bert-ptractice/blob/main/Bert_main_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch torchvision torchaudio




   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 34.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [ ]:
!pip install transformers datasets scikit-learn matplotlib seaborn


In [ ]:
import torch
print(torch.cuda.is_available())


False


In [ ]:
!pip install scikit-learn
!pip install tensorflow
!pip install matplotlib seaborn


In [ ]:
import torch

# Check if CUDA (GPU support) is available
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA device: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "No GPU detected.")


CUDA available: False
No GPU detected.


In [ ]:
# Step 1: Install datasets library if not already
!pip install datasets -q

# Step 2: Import required libraries
from datasets import load_dataset

# Step 3: Load CoNLL-03 dataset
print("\nLoading CoNLL-03 dataset...")
conll_dataset = load_dataset("conll2003")
print("CoNLL-03 loaded ✅")

# Step 4: Load ADE dataset
print("\nLoading ADE dataset...")
ade_dataset = load_dataset("ade_corpus_v2", "Ade_corpus_v2_classification")  # using classification version
print("ADE dataset loaded ✅")

# Step 5: Quick check: Print dataset structure
print("\nCoNLL-03 structure:")
print(conll_dataset)

print("\nADE structure:")
print(ade_dataset)



Loading CoNLL-03 dataset...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

NotImplementedError: Loading a dataset cached in a LocalFileSystem is not supported.

In [ ]:
from datasets import (
    load_dataset,
    DatasetDict,
    concatenate_datasets,
    Sequence,
    Value,
)
from transformers import AutoTokenizer
import re

# 1. Load datasets
conll = load_dataset('conll2003')
ade  = load_dataset('ade_corpus_v2', 'Ade_corpus_v2_classification', split='train')

# 2. Load tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')

# 3. Helper: clean ADE text
def clean_ade_text(text):
    text = text.replace('\n', ' ').replace('\t', ' ')
    return re.sub(' +', ' ', text).strip()

# 4. Preprocess functions
# 4a. CoNLL-03: convert IDs→strings, keep only tokens & ner_tags
conll_labels = conll['train'].features['ner_tags'].feature
def preprocess_conll(ex):
    return {
        'tokens':   ex['tokens'],
        'ner_tags': [conll_labels.int2str(int(t)) for t in ex['ner_tags']],
    }

# 4b. ADE: tokenize text, assign all 'O', keep only tokens & ner_tags
def preprocess_ade(ex):
    text   = clean_ade_text(ex['text'])
    tokens = tokenizer.tokenize(text)
    return {
        'tokens':   tokens,
        'ner_tags': ['O'] * len(tokens),
    }

# 5. Apply preprocessing & drop old columns
conll_cleaned = conll.map(
    preprocess_conll,
    remove_columns=['id','pos_tags','chunk_tags']
)
ade_cleaned  = ade.map(
    preprocess_ade,
    remove_columns=['text','label']
)

# 6. Cast ner_tags to Sequence of strings on both
conll_cleaned = conll_cleaned.cast_column('ner_tags', Sequence(Value('string')))
ade_cleaned   = ade_cleaned.cast_column(  'ner_tags', Sequence(Value('string')))

# 7. Combine into a single DatasetDict
combined_dataset = DatasetDict({
    'train':      concatenate_datasets([conll_cleaned['train'], ade_cleaned]),
    'validation': conll_cleaned['validation'],
    'test':       conll_cleaned['test'],
})

print("✅ Full Preprocessing done.")
print(combined_dataset)


In [ ]:
from transformers import AutoTokenizer
from datasets import DatasetDict
import numpy as np

# 1. Define your labels (must match CoNLL tag set)
label_list = ['O', 'B-PER', 'I-PER', 'B-LOC', 'I-LOC', 'B-ORG', 'I-ORG', 'B-MISC', 'I-MISC']
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

# 2. Reload tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')

# 3. Normalizer: turn any raw label into one of our strings
def normalize_label(raw):
    # If already in our label2id, great
    if raw in label2id:
        return raw
    # If integer or numeric string, map via id2label
    try:
        idx = int(raw)
        return id2label[idx]
    except:
        # fallback
        return 'O'

# 4. Tokenize + align labels
def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples['tokens'],
        is_split_into_words=True,
        padding='max_length',
        truncation=True,
        max_length=128,
    )
    all_labels = examples['ner_tags']
    aligned_labels = []

    for i, labels in enumerate(all_labels):
        word_ids = tokenized.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)  # special tokens
            else:
                raw_label = labels[word_idx]
                lab = normalize_label(raw_label)
                label_ids.append(label2id[lab])
            previous_word_idx = word_idx
        aligned_labels.append(label_ids)

    tokenized["labels"] = aligned_labels
    return tokenized

# 5. Apply to each split
encoded_dataset = DatasetDict()
for split in ['train', 'validation', 'test']:
    encoded = combined_dataset[split].map(
        tokenize_and_align_labels,
        batched=True,
        remove_columns=['tokens', 'ner_tags']
    )
    encoded_dataset[split] = encoded

print(encoded_dataset)
# Expect features: input_ids, attention_mask, token_type_ids (if any), labels


In [ ]:
from transformers import TrainingArguments, Trainer, AutoModelForTokenClassification

# 1. Load model
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

# 2. Simplified TrainingArguments
training_args = TrainingArguments(
    output_dir="./results",
    do_train=True,
    do_eval=True,
    logging_steps=50,
    save_steps=500,           # checkpoint every 500 steps
    save_total_limit=2,       # keep last 2
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
)

# 3. Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    tokenizer=tokenizer,
)

# 4. Train
trainer.train()

# 5. Evaluate
eval_results = trainer.evaluate()
print("Validation results:", eval_results)


In [ ]:
# Evaluate the model on the test set
eval_results = trainer.evaluate(encoded_dataset["test"])
print("Test results:", eval_results)


In [ ]:
!pip install --upgrade transformers


In [ ]:
from transformers import TrainingArguments, Trainer, AutoModelForTokenClassification, AutoTokenizer

# Load the saved model and tokenizer
model = AutoModelForTokenClassification.from_pretrained("BRET_projectmodel")
tokenizer = AutoTokenizer.from_pretrained("BRET_projectmodel")

# Define training arguments without using evaluation_strategy
training_args = TrainingArguments(
    output_dir="./results",
    eval_steps=500,  # Evaluate every 500 steps
    save_steps=500,  # Save checkpoint every 500 steps
    save_total_limit=2,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
)

# Define trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    tokenizer=tokenizer,
)

# Start training
trainer.train()

# Evaluate after training
eval_results = trainer.evaluate(encoded_dataset["test"])
print("Test results:", eval_results)


In [ ]:
from datasets import load_dataset, concatenate_datasets
from transformers import BertTokenizer, BertForTokenClassification, TrainingArguments, Trainer

# 1. Load the datasets
# Load FiNER-ORD (Financial NER) dataset
finer_ord = load_dataset("nlpaueb/finer-139")
print("FiNER-ORD Dataset:", finer_ord)

# Load the Wikipedia NER (WikiAnn) dataset (English version)
wikiann = load_dataset("wikiann", "en")
print("Wikipedia NER (WikiAnn) Dataset:", wikiann)

# 2. Preprocess the datasets

def preprocess_function(examples):
    # Tokenize the data
    tokenized_inputs = tokenizer(examples['tokens'], truncation=True, padding='max_length', is_split_into_words=True)

    # Align the labels
    labels = examples['ner_tags']

    # Ensure that labels are aligned with tokenized words (since tokenization might split words into subwords)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Load the tokenizer
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)

# Preprocess both datasets
finer_ord_encoded = finer_ord['train'].map(preprocess_function, batched=True)
wikiann_encoded = wikiann['train'].map(preprocess_function, batched=True)

# Concatenate the datasets
combined_encoded = concatenate_datasets([finer_ord_encoded, wikiann_encoded])

# 3. Load the pretrained BERT model for NER
model = BertForTokenClassification.from_pretrained(model_name, num_labels=len(finer_ord['train'].features['ner_tags'].unique()) + 1)
print("Model Loaded:", model)

# 4. Define the training arguments
training_args = TrainingArguments(
    output_dir="./results",            # output directory
    evaluation_strategy="epoch",       # Evaluate every epoch
    learning_rate=2e-5,                # Learning rate
    per_device_train_batch_size=8,     # Batch size for training
    per_device_eval_batch_size=8,      # Batch size for evaluation
    num_train_epochs=3,                # Number of training epochs
    weight_decay=0.01,                 # Weight decay for optimization
)

# 5. Initialize the Trainer
trainer = Trainer(
    model=model,                     # The model to train
    args=training_args,              # Training arguments
    train_dataset=combined_encoded,  # The training dataset
    eval_dataset=combined_encoded,   # The evaluation dataset
)

# 6. Start training
trainer.train()

# 7. Save the trained model
model.save_pretrained("./trained_model")
tokenizer.save_pretrained("./trained_model")

print("Model trained and saved.")


In [ ]:
from datasets import load_dataset

# Load FiNER-ORD (Financial NER) dataset
finer_ord = load_dataset("nlpaueb/finer-139")
print("FiNER-ORD Dataset:", finer_ord)

# Load the Wikipedia NER (WikiAnn) dataset (English version)
wikiann = load_dataset("wikiann", "en")
print("Wikipedia NER (WikiAnn) Dataset:", wikiann)


In [ ]:
from transformers import BertTokenizer

# Define the common label set
common_labels = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']

# Load the tokenizer
model_name = "BRET_projectmodel"  # Use your custom model name
tokenizer = BertTokenizer.from_pretrained(model_name)

def preprocess_function(examples, dataset_name):
    # Tokenize the data
    tokenized_inputs = tokenizer(examples['tokens'], truncation=True, padding='max_length', is_split_into_words=True)

    # Standardize ner_tags for both datasets
    if dataset_name == 'finer-ord':
        # Map the FNSP-2 (FiNER-ORD) labels to the common label set
        ner_tags = examples['ner_tags']
        labels = [common_labels.index(label) if label in common_labels else 0 for label in ner_tags]
    else:
        # Wikipedia NER labels are already in common format, so we can directly use them
        labels = examples['ner_tags']

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Preprocess FiNER-ORD dataset
finer_ord_encoded = finer_ord['train'].map(lambda examples: preprocess_function(examples, 'finer-ord'), batched=True)

# Preprocess Wikipedia NER dataset
wikiann_encoded = wikiann['train'].map(lambda examples: preprocess_function(examples, 'wikiann'), batched=True)


In [ ]:
def preprocess_function(examples):
    tokenized_inputs = tokenizer(
        examples['tokens'],
        truncation=True,
        padding='max_length',
        max_length=128,
        is_split_into_words=True,
    )

    labels = []
    for i, label in enumerate(examples['ner_tags']):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        previous_word_idx = None
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(label[word_idx] if label[word_idx] != 0 else -100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs


In [ ]:
from transformers import BertTokenizerFast
tokenizer = BertTokenizerFast.from_pretrained('bert-base-cased')


# Preprocess FiNER-ORD
finer_ord_encoded = finer_ord['train'].map(
    preprocess_function,
    batched=True,
    num_proc=4,           # 4 CPU cores used 🚀 (adjust based on your machine)
    remove_columns=finer_ord['train'].column_names,
    desc="Preprocessing FiNER-ORD"
)

# Preprocess WikiAnn


In [ ]:
wikiann_encoded = wikiann['train'].map(
    preprocess_function,
    batched=True,
    num_proc=4,
    remove_columns=wikiann['train'].column_names,
    desc="Preprocessing WikiAnn"
)


In [ ]:
from datasets import concatenate_datasets

combined_encoded = concatenate_datasets([finer_ord_encoded, wikiann_encoded])

print("✅ Combined dataset size:", len(combined_encoded))


In [ ]:
!pip install evaluate


In [ ]:
!pip install seqeval


In [ ]:
# prompt: check transformers version

import transformers

transformers.__version__


In [ ]:
common_labels = ['O','B-PER','I-PER','B-ORG','I-ORG','B-LOC','I-LOC']
label2id = {l:i for i,l in enumerate(common_labels)}
id2label = {i:l for l,i in label2id.items()}


In [ ]:
from datasets import load_dataset

finer = load_dataset("nlpaueb/finer-139")
wiki  = load_dataset("wikiann","en")
print(f"FiNER-ORD splits: {list(finer.keys())}, WikiAnn splits: {list(wiki.keys())}")


In [ ]:
from transformers import BertTokenizerFast
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")


In [ ]:
def tokenize_and_align_labels(examples, dataset_name):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
    )
    labels = []

    word_ids = tokenized_inputs.word_ids()
    previous_word_idx = None

    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)
        elif word_idx != previous_word_idx:
            labels.append(label2id[examples["ner_tags"][word_idx]])
        else:
            labels.append(label2id[examples["ner_tags"][word_idx]])
        previous_word_idx = word_idx

    tokenized_inputs["labels"] = labels
    return tokenized_inputs


In [ ]:
# Build a batch of size 1
sample_batch = {
    "tokens":   [ finer["train"][0]["tokens"] ],
    "ner_tags": [ finer["train"][0]["ner_tags"] ]
}

print(tokenize_and_align_labels(sample_batch, "finer"))


TypeError: unhashable type: 'list'

In [ ]:
# FiNER-ORD dataset
finer_train = finer["train"].map(
    lambda ex: tokenize_and_align_labels(ex, "finer"),
    batched=True,
    num_proc=4,
    remove_columns=finer["train"].column_names,
)





In [ ]:
wiki_train = wiki["train"].map(
    lambda ex: tokenize_and_align_labels(ex, "wiki"),
    batched=True,
    num_proc=4,

    remove_columns=wiki["train"].column_names,
)# WikiAnn dataset

Map (num_proc=4):   0%|          | 0/20000 [00:00<?, ? examples/s]

TypeError: unhashable type: 'list'

In [ ]:
# Check a few samples from the tokenized FiNER-ORD dataset
print(finer_train[0])




In [ ]:
# Check a few samples from the tokenized WikiAnn dataset
print(wiki_train[0])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Define the paths where you want to save the data on your Google Drive
finer_train_path = "/content/drive/MyDrive/finer_train"


# Save the datasets to your Google Drive
finer_train.save_to_disk(finer_train_path)



In [ ]:
wiki_train_path = "/content/drive/MyDrive/wiki_train"
wiki_train.save_to_disk(wiki_train_path)

In [ ]:
from transformers import BertForTokenClassification, Trainer, TrainingArguments

# Load pre-trained model (e.g., BERT)
model = BertForTokenClassification.from_pretrained("bert-base-uncased", num_labels=num_labels)

# Define Training Arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    logging_dir='./logs',
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=finer_train,
    eval_dataset=wiki_train,
)


In [ ]:
trainer.train()


In [ ]:
import torch
from transformers import AutoModelForTokenClassification, AutoTokenizer

# 1. Load your fine-tuned model & tokenizer
model = AutoModelForTokenClassification.from_pretrained("BRET_projectmodel")
tokenizer = AutoTokenizer.from_pretrained("BRET_projectmodel")

# 2. Build a simple map from tail labels to human types
entity_map = {
    'PER': 'PERSON',
    'LOC': 'LOCATION',
    'ORG': 'ORGANIZATION',
    'MISC': 'MISCELLANEOUS',
    'O': 'O'
}

# 3. Utility to group sub-words and their labels into full words
def group_tokens_labels(tokens, labels):
    words, word_labels = [], []
    current_word, current_label = "", None

    for t, lab in zip(tokens, labels):
        if t in tokenizer.all_special_tokens:
            continue
        if t.startswith("##"):
            # subword continuation
            current_word += t[2:]
        else:
            # push previous
            if current_word:
                words.append(current_word)
                word_labels.append(current_label)
            current_word = t
            current_label = lab
    # push last
    if current_word:
        words.append(current_word)
        word_labels.append(current_label)

    return words, word_labels

# 4. Inference function
def predict(text):
    # tokenize & predict
    inputs = tokenizer(text, return_tensors="pt", truncation=True)
    with torch.no_grad():
        logits = model(**inputs).logits
    preds = torch.argmax(logits, dim=-1)[0].tolist()
    toks  = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    # map ids → label strings via model config
    labs = [model.config.id2label[p] for p in preds]

    # group into words
    words, word_labels = group_tokens_labels(toks, labs)

    # map B-/I- → high-level type
    final = []
    for lab in word_labels:
        if lab == "O":
            final.append("O")
        else:
            _, tail = lab.split("-", 1)
            final.append(entity_map.get(tail, "O"))
    return words, final

# 5. Interactive loop
print("NER> Type any sentence and see entities. (‘exit’ or ‘quit’ to stop.)")
while True:
    text = input(">> ")
    if text.lower() in {"exit", "quit"}:
        break
    words, types = predict(text)
    print("\nWord\t→ Entity")
    print("-" * 30)
    for w, t in zip(words, types):
        print(f"{w}\t→ {t}")
    print()


NER> Type any sentence and see entities. (‘exit’ or ‘quit’ to stop.)
>> apple is a compny

Word	→ Entity
------------------------------
apple	→ O
is	→ O
a	→ O
compny	→ O

>> apple launched a new iphone in california

Word	→ Entity
------------------------------
apple	→ O
launched	→ O
a	→ O
new	→ O
iphone	→ O
in	→ O
california	→ O



KeyboardInterrupt: Interrupted by user

In [ ]:
import torch
from transformers import AutoModelForTokenClassification, AutoTokenizer

# Load the saved model and tokenizer
model = AutoModelForTokenClassification.from_pretrained("./BRET_projectmodel")
tokenizer = AutoTokenizer.from_pretrained("./BRET_projectmodel")

# Define the label mapping
entity_map = {
    'PER': 'PERSON',
    'LOC': 'LOCATION',
    'ORG': 'ORGANIZATION',
    'MISC': 'MISCELLANEOUS',
    'O': 'O'
}

def group_tokens_labels(tokens, labels):
    words, word_labels = [], []
    current_word, current_label = "", None
    for t, lab in zip(tokens, labels):
        if t in tokenizer.all_special_tokens:
            continue
        if t.startswith("##"):  # Handle subwords
            current_word += t[2:]
        else:
            if current_word:
                words.append(current_word)
                word_labels.append(current_label)
            current_word = t
            current_label = lab
    if current_word:
        words.append(current_word)
        word_labels.append(current_label)
    return words, word_labels

def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True)
    with torch.no_grad():
        logits = model(**inputs).logits
    preds = torch.argmax(logits, dim=-1)[0].tolist()
    toks = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    labs = [model.config.id2label[p] for p in preds]

    # Group subwords into full words with their labels
    words, word_labels = group_tokens_labels(toks, labs)

    # Map labels to appropriate entities
    final = [entity_map.get(lab.split("-")[1], "O") if lab != "O" else "O" for lab in word_labels]

    return words, final

# Example text to predict
input_text = "Apple launched a new iPhone in California"
words, entities = predict(input_text)

print("\nWord\t→ Entity")
print("-" * 30)
for w, e in zip(words, entities):
    print(f"{w}\t→ {e}")



Word	→ Entity
------------------------------
Apple	→ LOCATION
launched	→ O
a	→ O
new	→ O
iPhone	→ MISCELLANEOUS
in	→ O
California	→ ORGANIZATION


In [ ]:
from sklearn.metrics import precision_recall_fscore_support
import numpy as np
from transformers import AutoModelForTokenClassification, AutoTokenizer
from datasets import load_dataset
from transformers import Trainer, TrainingArguments

# Load the model and tokenizer (from your saved directory)
model = AutoModelForTokenClassification.from_pretrained("./BRET_projectmodel")  # Replace with your model path
tokenizer = AutoTokenizer.from_pretrained("./BRET_projectmodel")  # Replace with your tokenizer path

# Load your dataset (replace 'conll2003' with the actual dataset you used for training)
dataset = load_dataset('conll2003')  # Or load your specific dataset

# Tokenize and encode the dataset
def preprocess_function(examples):
    # Tokenize with padding and truncation
    tokenized_inputs = tokenizer(examples['tokens'], truncation=True, padding='max_length', is_split_into_words=True, max_length=128)
    labels = examples['ner_tags']

    # Adjust labels to match the tokenized sequence length
    # Padding tokens should have label -100
    labels = [label + [-100] * (128 - len(label)) if len(label) < 128 else label for label in labels]

    tokenized_inputs['labels'] = labels
    return tokenized_inputs

encoded_dataset = dataset.map(preprocess_function, batched=True)

# Define the evaluation function to calculate precision, recall, f1, and domain-based accuracy
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(axis=-1)

    # Flatten the lists
    labels = labels.flatten()
    preds = preds.flatten()

    # Remove ignored tokens (e.g., padding tokens)
    mask = labels != -100  # Ignore -100 for padding tokens
    labels = labels[mask]
    preds = preds[mask]

    # Calculate precision, recall, F1 for each label
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average=None, labels=np.unique(labels))

    # Calculate domain-based accuracy (percentage of correctly predicted entities)
    domain_accuracy = np.sum(labels == preds) / len(labels)

    metrics = {
        'precision': precision.tolist(),
        'recall': recall.tolist(),
        'f1': f1.tolist(),
        'domain_accuracy': domain_accuracy
    }
    return metrics

# Define evaluation arguments (remove evaluation_strategy)
training_args = TrainingArguments(
    output_dir='./NER_model',  # Output directory for saving model checkpoints and logs
    per_device_eval_batch_size=8,  # Adjust based on your system's memory
    num_train_epochs=3,  # Adjust this depending on your training configuration
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    eval_dataset=encoded_dataset['test'],  # Replace with your test set
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# Perform evaluation
results = trainer.evaluate()

# Print the evaluation results in the terminal
print("Evaluation Results:")
print(f"Precision: {results['eval_precision']}")
print(f"Recall: {results['eval_recall']}")
print(f"F1-Score: {results['eval_f1']}")
print(f"Domain-based Accuracy: {results['eval_domain_accuracy']}")


Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

<ipython-input-50-797b81a99bd1>:65: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Trainer is attempting to log a value of "[0.844795828728499, 0.048539518900343644, 0.0679304897314376, 0.02484055052030883, 0.11473880597014925, 0.05531385954008701, 0.06818181818181818, 0.020524515393386546, 0.06796116504854369]" of type <class 'list'> for key "eval/precision" as a scalar. This invocation of Tensorboard's writer.add_scalar() is incorrect so we dropped this attribute.
Trainer is attempting to log a value of "[0.7779140463951152, 0.06988249845392702, 0.1115916955017301, 0.044551475015051176, 0.1473053892215569, 0.053357314148681056, 0.04669260700389105, 0.02564102564102564, 0.06481481481481481]" of type <class 'list'> for key "eval/recall" as a scalar. This invocation of Tensorboard's writer.add_scalar() is incorrect so we dropped this attribute.
Trainer is attempting to log a value of "[0.8099766342444167, 0.05728770595690748, 0.08445171849427169, 0.03189655172413793, 0.12899842684845306, 0.05431797375648459, 0.05542725173210162, 0.022799240025332488, 0.066350710900473

Evaluation Results:
Precision: [0.844795828728499, 0.048539518900343644, 0.0679304897314376, 0.02484055052030883, 0.11473880597014925, 0.05531385954008701, 0.06818181818181818, 0.020524515393386546, 0.06796116504854369]
Recall: [0.7779140463951152, 0.06988249845392702, 0.1115916955017301, 0.044551475015051176, 0.1473053892215569, 0.053357314148681056, 0.04669260700389105, 0.02564102564102564, 0.06481481481481481]
F1-Score: [0.8099766342444167, 0.05728770595690748, 0.08445171849427169, 0.03189655172413793, 0.12899842684845306, 0.05431797375648459, 0.05542725173210162, 0.022799240025332488, 0.06635071090047394]
Domain-based Accuracy: 0.6543340152901906


In [ ]:
!pip install --upgrade transformers


In [ ]:
from transformers import Trainer, TrainingArguments, BertTokenizer, BertForSequenceClassification
from datasets import load_dataset

# Load dataset and tokenizer
dataset = load_dataset("imdb")  # Example dataset
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Tokenize the dataset with padding and truncation
def tokenize_function(examples):
    return tokenizer(
        examples['text'],  # Replace with your actual data column
        padding=True,      # Pads to the longest sequence in the batch or max_length if set
        truncation=True,   # Truncates sequences that exceed max_length
        max_length=512,    # You can adjust the max_length as needed
        return_tensors="pt"  # Returns pytorch tensors for compatibility with the model
    )

# Tokenizing the datasets
encoded_datasets = dataset.map(tokenize_function, batched=True)

# Prepare the dataset for training
train_dataset = encoded_datasets["train"]
eval_dataset = encoded_datasets["test"]

# Load the model for sequence classification (BERT-based in this case)
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

# Define the training arguments
training_args = TrainingArguments(
    output_dir="./results",            # output directory
    evaluation_strategy="epoch",       # Evaluate every epoch
    learning_rate=2e-5,                # Learning rate
    per_device_train_batch_size=8,     # Training batch size
    per_device_eval_batch_size=8,      # Evaluation batch size
    num_train_epochs=3,                # Number of training epochs
    weight_decay=0.01,                 # Strength of weight decay
    logging_dir='./logs',              # Directory for storing logs
    logging_steps=10,                  # Log every 10 steps
    save_steps=1000,                   # Save model every 1000 steps
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

# Train and evaluate
trainer.train()

# Evaluate the model on the test set
test_metrics = trainer.evaluate(eval_dataset=eval_dataset)
print(f"\n→ Test results:", test_metrics)


ModuleNotFoundError: No module named 'datasets'

AFTER EVERY RUNTIME ENDS WE NEED TO RESTABLISH THE PATHS

In [ ]:
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


In [ ]:
import shutil

# Define source and destination paths
source_path = '/content/drive/MyDrive/BRET_projectmodel'
destination_path = '/content/BRET_projectmodel'

# Copy model folder from Google Drive to local runtime path
shutil.copytree(source_path, destination_path)


'/content/BRET_projectmodel'

In [ ]:
# Define the path in Google Drive where you want to save the model and tokenizer
save_path = '/content/drive/MyDrive/BRET_projectmodel'

# Save the model and tokenizer
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"Model and tokenizer saved to: {save_path}")


Model and tokenizer saved to: /content/drive/MyDrive/BRET_projectmodel


In [ ]:
from transformers import AutoModelForTokenClassification, AutoTokenizer

# Load the model and tokenizer from the saved location
model = AutoModelForTokenClassification.from_pretrained(save_path)
tokenizer = AutoTokenizer.from_pretrained(save_path)

print("Model and tokenizer loaded successfully.")


Model and tokenizer loaded successfully.


In [ ]:
# Save the trained model
model.save_pretrained("BRET_projectmodel")

# Save the tokenizer
tokenizer.save_pretrained("BRET_projectmodel")


('BRET_projectmodel/tokenizer_config.json',
 'BRET_projectmodel/special_tokens_map.json',
 'BRET_projectmodel/vocab.txt',
 'BRET_projectmodel/added_tokens.json',
 'BRET_projectmodel/tokenizer.json')